In [1]:
# Company payroll policy (static context)
company_policy = {
    "country": "India",
    "pf_rate": 0.12,
    "esi_rate": 0.0075,
    "esi_limit": 21000,
    "tax_regime": "New Regime"
}

company_policy


{'country': 'India',
 'pf_rate': 0.12,
 'esi_rate': 0.0075,
 'esi_limit': 21000,
 'tax_regime': 'New Regime'}

In [2]:
def detect_role(query):
    if "my" in query.lower() or "i am" in query.lower():
        return "EMPLOYEE"
    return "HR"

# Test
detect_role("Why is my PF deducted?")


'EMPLOYEE'

In [3]:
def build_prompt(role, query):
    if role == "EMPLOYEE":
        return f"""
        You are a Payroll Assistant.
        Explain payroll concepts in simple language.
        Avoid legal advice.
        Question: {query}
        """
    else:
        return f"""
        You are an HR Payroll Expert.
        Use formal tone and legal justification.
        Question: {query}
        """

# Test
print(build_prompt("HR", "Explain PF deduction"))



        You are an HR Payroll Expert.
        Use formal tone and legal justification.
        Question: Explain PF deduction
        


In [4]:
def validate_query(query):
    allowed_topics = ["pf", "esi", "tax", "salary", "payslip", "bonus"]
    return any(topic in query.lower() for topic in allowed_topics)

# Test
validate_query("Tell me about company profit")


False

In [5]:
def payroll_response(query, salary=50000):
    if not validate_query(query):
        return "I am not authorized to answer this non-payroll question."

    if "pf" in query.lower():
        pf = salary * company_policy["pf_rate"]
        return f"Provident Fund is deducted at 12% as per EPF Act. Amount: ₹{pf}"

    if "esi" in query.lower():
        if salary <= company_policy["esi_limit"]:
            esi = salary * company_policy["esi_rate"]
            return f"ESI is applicable. Deduction: ₹{esi}"
        else:
            return "ESI is not applicable for this salary."

    return "Payroll information provided as per company policy."


In [6]:
import re

def redact_sensitive_data(text):
    text = re.sub(r'\b\d{12}\b', 'XXXX-XXXX-XXXX', text)
    text = re.sub(r'\b\d{10}\b', 'XXXXXXXXXX', text)
    return text

# Test
redact_sensitive_data("Employee Aadhaar 123412341234 Phone 9876543210")


'Employee Aadhaar XXXX-XXXX-XXXX Phone XXXXXXXXXX'

In [7]:
def payroll_ai_copilot(query, salary=50000):
    role = detect_role(query)
    prompt = build_prompt(role, query)
    response = payroll_response(query, salary)
    safe_response = redact_sensitive_data(response)

    return {
        "Role": role,
        "Prompt Used": prompt.strip(),
        "AI Response": safe_response
    }

# Test
payroll_ai_copilot("Why is my PF deducted?", 60000)


{'Role': 'EMPLOYEE',
 'Prompt Used': 'You are a Payroll Assistant.\n        Explain payroll concepts in simple language.\n        Avoid legal advice.\n        Question: Why is my PF deducted?',
 'AI Response': 'Provident Fund is deducted at 12% as per EPF Act. Amount: ₹7200.0'}

In [8]:
queries = [
    "Why is my PF deducted?",
    "Can PF be avoided?",
    "Is ESI applicable to me?",
    "Explain salary breakup",
    "Why has my tax increased?",
    "Is bonus taxable?",
    "What is gratuity?",
    "Payslip mismatch reason",
    "Overtime taxable?",
    "Why is salary on hold?"
]

for q in queries:
    print(q)
    print(payroll_ai_copilot(q)["AI Response"])
    print("-"*50)


Why is my PF deducted?
Provident Fund is deducted at 12% as per EPF Act. Amount: ₹6000.0
--------------------------------------------------
Can PF be avoided?
Provident Fund is deducted at 12% as per EPF Act. Amount: ₹6000.0
--------------------------------------------------
Is ESI applicable to me?
ESI is not applicable for this salary.
--------------------------------------------------
Explain salary breakup
Payroll information provided as per company policy.
--------------------------------------------------
Why has my tax increased?
Payroll information provided as per company policy.
--------------------------------------------------
Is bonus taxable?
Payroll information provided as per company policy.
--------------------------------------------------
What is gratuity?
I am not authorized to answer this non-payroll question.
--------------------------------------------------
Payslip mismatch reason
Payroll information provided as per company policy.
-------------------------------